# SymplecticGKP — Interactive Examples

This notebook demonstrates the key functionality of the `SymplecticGKP`
package through worked examples relevant to multi-mode
Gottesman-Kitaev-Preskill (GKP) codes.

**Setup:** Make sure you activate the SymplecticGKP environment before
running. In the first cell we do this programmatically.

In [ ]:
using Pkg
Pkg.activate(dirname(@__DIR__))   # the SymplecticGKP package env (repo root)
using SymplecticGKP
using LinearAlgebra

---
## 1. The Symplectic Gram Matrix

A GKP code on $n$ bosonic modes is defined by a lattice generator
matrix $M \in \mathbb{R}^{2n \times 2n}$. The **symplectic Gram matrix**

$$A = M \, J_{2n} \, M^\top, \qquad J_{2n} = \begin{pmatrix} 0 & I_n \\ -I_n & 0 \end{pmatrix}$$

is anti-symmetric and integer-valued for a valid GKP code. It encodes
the commutation relations of the displacement stabilizers.

Let's build a simple example: a 2-mode code ($n=2$) from an SIS-type
lattice with $q = 5$.

In [ ]:
n = 2
q = 5
H = [0 1; 1 0]   # symmetric matrix mod q

# Standard symplectic form J_{2n}
J = [zeros(Int, n, n)       Matrix{Int}(I, n, n);
    -Matrix{Int}(I, n, n)   zeros(Int, n, n)]

# SIS generator matrix
M_H = [Matrix{Int}(I, n, n)   H;
       zeros(Int, n, n)        q * Matrix{Int}(I, n, n)]

# Symplectic Gram matrix
A = M_H * J * M_H'

println("Generator matrix M_H:")
display(M_H)
println("\nSymplectic Gram matrix A = M_H * J * M_H':")
display(A)
println("\nA is anti-symmetric: ", A == -A')
println("Logical dimension d² = |det(A)| = ", abs(det(A)))

---
## 2. Computing the Canonical Form

The function `symplectic_basis_over_ZZ(A)` finds a unimodular matrix
$C$ (i.e., $|\det C| = 1$) such that

$$F = C \, A \, C^\top = \begin{pmatrix} 0 & D \\ -D & 0 \end{pmatrix}$$

where $D = \mathrm{diag}(d_1, d_2, \ldots, d_n)$ with the **divisibility**
condition $d_1 \mid d_2 \mid \cdots \mid d_n$.

These $d_j$ are the **invariant factors** — they are uniquely determined
by the lattice and classify the code up to integer basis changes.

In [ ]:
F, C = symplectic_basis_over_ZZ(A)

println("Canonical form F = C * A * C':")
display(F)
println("\nUnimodular transform C:")
display(C)
println("\ndet(C) = ", round(Int, det(C)))

inv_factors = extract_invariant_factors(F)
println("\nInvariant factors: ", inv_factors)
println("Divisibility check: ", all(inv_factors[i+1] % inv_factors[i] == 0
                                     for i in 1:length(inv_factors)-1))

# Verify
println("\nC * A * C' == F ? ", C * A * C' == F)

---
## 3. Understanding Invariant Factors

The invariant factors tell us the **structure** of the GKP code:

| Invariant factors | Meaning |
|---|---|
| $(d, d, \ldots, d)$ | Uniform: each mode encodes $d$ logical levels |
| $(1, 1, \ldots, 1, d^n)$ | All encoding concentrated in one mode |
| $(1, d)$ for prime $d$ | Single qudit of dimension $d$ in 2 modes |

Let's look at several examples.

In [ ]:
examples = [
    ("Uniform qubit code (n=3)",
     [0 0 0 2 0 0; 0 0 0 0 2 0; 0 0 0 0 0 2;
     -2 0 0 0 0 0; 0 -2 0 0 0 0; 0 0 -2 0 0 0]),

    ("Concentrated code (n=2)",
     [0 0 1 0; 0 0 0 10; -1 0 0 0; 0 -10 0 0]),

    ("Prime dimension d=7 (n=2)",
     [0 0 1 0; 0 0 0 7; -1 0 0 0; 0 -7 0 0]),

    ("Dense coupling (n=2)",
     [0 5 3 0; -5 0 0 2; -3 0 0 7; 0 -2 -7 0]),
]

for (name, A_ex) in examples
    A_mat = Matrix{Int}(A_ex)
    F_ex, _ = symplectic_basis_over_ZZ(A_mat)
    d_ex = extract_invariant_factors(F_ex)
    d_total = prod(d_ex)
    println("$name")
    println("  Invariant factors: $d_ex")
    println("  Total logical dimension: d = $d_total")
    println("  Logical qubits: k = $(round(log2(d_total), digits=2))")
    println()
end

---
## 4. Pfaffian Divisors — The True Invariant

The **Pfaffian divisors** $(\pi_1, \ldots, \pi_n)$ are the finest
invariant of the anti-symmetric form under unimodular congruence.
Two block-diagonal forms $D$ and $D'$ are related by a unimodular
transform if and only if they have the same Pfaffian divisors.

The Smith invariant factors enforce divisibility, but there may be
multiple valid block-diagonal decompositions with the same Pfaffian
divisors.

In [ ]:
# (5, 2) and (10, 1) have the same Pfaffian divisors
d_a = [5, 2]
d_b = [10, 1]

pi_a = pfaffian_divisors(d_a)
pi_b = pfaffian_divisors(d_b)

println("D = (5, 2):  Pfaffian divisors = $pi_a")
println("D = (10, 1): Pfaffian divisors = $pi_b")
println("Equivalent? ", verify_equivalence(d_a, d_b))

println()

# (2, 2, 2) and (1, 1, 8) have DIFFERENT Pfaffian divisors
d_c = [2, 2, 2]
d_d = [1, 1, 8]

pi_c = pfaffian_divisors(d_c)
pi_d = pfaffian_divisors(d_d)

println("D = (2, 2, 2): Pfaffian divisors = $pi_c")
println("D = (1, 1, 8): Pfaffian divisors = $pi_d")
println("Equivalent? ", verify_equivalence(d_c, d_d))

---
## 5. Balanced Diagonal — Minimizing max(dⱼ)

Given the invariant factors from the Smith canonical form, we can
often find an alternative diagonal decomposition that **minimizes
the largest entry** $\max_j d_j$. This improves the GKP code's
normal-form distance.

The key idea: for each prime $p$, the multiset of $p$-adic valuations
is fixed. But different primes can be **anti-correlated** to spread
the entries more evenly.

In [ ]:
test_cases = [
    [1, 10],
    [1, 36],
    [1, 12],
    [1, 2, 18],
    [1, 900],
    [1, 1, 8],     # cannot improve: single prime structure
    [2, 2, 2],     # already uniform
    [5, 5],        # already uniform
]

println("Invariant factors → Balanced diagonal")
println("-" ^ 55)
for e in test_cases
    d = balanced_diagonal(e)
    improved = maximum(d) < maximum(e)
    status = improved ? "  ✓ improved" : "  (unchanged)"
    println("  $e → $d   max: $(maximum(e)) → $(maximum(d))$status")
end

---
## 6. End-to-End: balanced_gkp_form

The function `balanced_gkp_form(A)` combines everything:

1. Compute canonical form and invariant factors
2. Find balanced diagonal
3. Compute unimodular transform $U$ such that $U A U^\top = A_{\rm balanced}$

This is the main function you would use in practice.

In [ ]:
# A code with invariant factors (1, 36) — very unbalanced
A_unbalanced = [0 0 1 0; 0 0 0 36; -1 0 0 0; 0 -36 0 0]

d_bal, U, A_bal = balanced_gkp_form(A_unbalanced)

println("Original A:")
# display(A_unbalanced)
println(A_unbalanced)
println("\nBalanced A:")
# display(A_bal)
println(A_bal)
println("\nUnimodular transform U:")
display(U)
println("\ndet(U) = ", round(Int, det(U)))
println("U * A * U' == A_bal ? ", U * A_unbalanced * U' == A_bal)

println("\n--- Summary ---")
original_factors = extract_invariant_factors(
    symplectic_basis_over_ZZ(A_unbalanced)[1])
println("Original invariant factors: $original_factors  (max = $(maximum(original_factors)))")
println("Balanced diagonal:          $d_bal  (max = $(maximum(d_bal)))")

---
## 7. Impact on GKP Code Distance

The normal-form distance of a GKP code with diagonal $D$ is

$$\Delta_{\rm normal} = \frac{1}{\sqrt{\max_j D_{jj}}}$$

Balancing the diagonal directly improves this quantity.

In [ ]:
println("Code distance improvement from balancing")
println("=" ^ 60)
println()

cases = [
    ("2-mode, d=10",  [1, 10]),
    ("2-mode, d=36",  [1, 36]),
    ("2-mode, d=12",  [1, 12]),
    ("3-mode, d=36",  [1, 2, 18]),
    ("2-mode, d=900", [1, 900]),
]

for (name, e) in cases
    d = balanced_diagonal(e)
    delta_old = 1.0 / sqrt(maximum(e))
    delta_new = 1.0 / sqrt(maximum(d))
    improvement = (delta_new / delta_old - 1) * 100

    println("$name")
    println("  Smith form:    D = $e")
    println("  Balanced form: D = $d")
    println("  Δ_normal: $(round(delta_old, digits=4)) → $(round(delta_new, digits=4))")
    println("  Improvement: +$(round(improvement, digits=1))%")
    println()
end

---
## 8. Visualizing the Lattice Structure

For a 1-mode GKP code ($n=1$), the lattice lives in $\mathbb{R}^2$
and we can plot it directly. Let's visualize a single-mode square
GKP code and its dual.

In [ ]:
# Simple text-based lattice visualization for a 1-mode code
function print_lattice_2d(M::Matrix, title::String; radius=3)
    println("\n$title")
    println("Generator matrix:")
    display(M)

    # Generate lattice points
    points = Tuple{Float64, Float64}[]
    for a in -radius:radius, b in -radius:radius
        v = [a, b]' * M
        push!(points, (v[1], v[2]))
    end

    # Simple ASCII grid
    grid_size = 21
    center = (grid_size + 1) ÷ 2
    grid = fill('.', grid_size, grid_size)
    scale = 2.5

    for (x, y) in points
        gx = round(Int, x / scale * (center - 1)) + center
        gy = round(Int, -y / scale * (center - 1)) + center
        if 1 <= gx <= grid_size && 1 <= gy <= grid_size
            grid[gy, gx] = 'O'
        end
    end

    for row in 1:grid_size
        println("  ", join(grid[row, :], ' '))
    end
end

# Square GKP code: M = sqrt(2) * I
M_square = [sqrt(2) 0; 0 sqrt(2)]
print_lattice_2d(M_square, "Square GKP code (λ=2)", radius=4)

# Rectangular GKP code
M_rect = [1.0 0; 0 2.0]
print_lattice_2d(M_rect, "Rectangular GKP code", radius=4)

---
## 9. Building GKP Codes from SIS Lattices

The SIS (Short Integer Solutions) construction generates
$q$-symplectic lattices that yield good GKP codes.

Given a symmetric matrix $H \in \mathbb{Z}_q^{n \times n}$:

$$M_H = \begin{pmatrix} I_n & H \\ 0 & qI_n \end{pmatrix}$$

Then $A = M_H J M_H^\top = qJ$, giving uniform invariant factors
$(q, q, \ldots, q)$.

In [ ]:
println("SIS-based GKP codes")
println("=" ^ 50)

for (n_modes, q_val) in [(2, 3), (2, 5), (2, 7), (3, 5)]
    H_rand = rand(0:q_val-1, n_modes, n_modes)
    H_sym = (H_rand + H_rand') .% q_val   # make symmetric

    J_n = [zeros(Int, n_modes, n_modes)       Matrix{Int}(I, n_modes, n_modes);
          -Matrix{Int}(I, n_modes, n_modes)   zeros(Int, n_modes, n_modes)]

    M_sis = [Matrix{Int}(I, n_modes, n_modes)    H_sym;
             zeros(Int, n_modes, n_modes)         q_val * Matrix{Int}(I, n_modes, n_modes)]

    A_sis = M_sis * J_n * M_sis'
    F_sis, _ = symplectic_basis_over_ZZ(A_sis)
    d_sis = extract_invariant_factors(F_sis)

    println("n=$n_modes, q=$q_val:  invariant factors = $d_sis  " *
            "(uniform = $(all(d_sis .== q_val)))")
end

---
## 10. Equivalence Classes — Counting Distinct Codes

For a given logical dimension $d = \prod_j d_j$, the number of
**symplectically inequivalent** GKP codes (up to Gaussian unitaries)
equals the number of distinct invariant-factor sequences
$d_1 | d_2 | \cdots | d_n$ with $\prod d_j = d$.

But within each invariant-factor class, there may be **multiple valid
block-diagonal forms** related by unimodular congruence. The balanced
form picks the one minimizing the maximum entry.

In [ ]:
function count_divisor_chains(d_total::Int, n_modes::Int)
    """Count sequences (d1,...,dn) with d1|d2|...|dn and prod = d_total."""
    if n_modes == 1
        return [[d_total]]
    end
    chains = Vector{Int}[]
    for d1 in 1:d_total
        d_total % d1 == 0 || continue
        for sub in count_divisor_chains(d_total ÷ d1, n_modes - 1)
            if d1 <= sub[1]  # maintain divisibility ordering
                if sub[1] % d1 == 0  # d1 | d2
                    push!(chains, [d1; sub])
                end
            end
        end
    end
    return chains
end

println("Inequivalent GKP code classes by logical dimension")
println("=" ^ 55)

for (n_modes, d_total) in [(2, 4), (2, 6), (2, 8), (2, 10),
                            (2, 12), (3, 8), (3, 12)]
    chains = count_divisor_chains(d_total, n_modes)
    println("\nn=$n_modes modes, d=$d_total:")
    println("  $(length(chains)) inequivalent class(es):")
    for ch in chains
        d_b = balanced_diagonal(ch)
        tag = d_b == ch ? "" : "  → balanced: $d_b"
        println("    $ch$tag")
    end
end

---
## Summary

| Function | What it does |
|---|---|
| `symplectic_basis_over_ZZ(A)` | Canonical form with divisibility: $F = CAC^\top$ |
| `extract_invariant_factors(F)` | Read off $(d_1, \ldots, d_n)$ from $F$ |
| `pfaffian_divisors(d)` | Compute Pfaffian invariants of a diagonal |
| `verify_equivalence(d, d')` | Check if two diagonals are congruent |
| `balanced_diagonal(e)` | Find $D$ minimizing $\max d_j$ |
| `balanced_gkp_form(A)` | End-to-end: balanced form + transform |
| `symplectic_gram(d)` | Build $J_2 \otimes \mathrm{diag}(d)$ |

The key insight: the Smith canonical form (with divisibility) is the
**unique** invariant, but it is not always the most **useful**
block-diagonal form. The balanced form minimizes $\max d_j$, which
directly improves the normal-form distance of the GKP code.